In [38]:
def open():
    """ This is to open Gift. """
    print("open gift")
open()

open gift


In [39]:
print(open.__name__)
print(open.__doc__)
print(open.__module__)

open
This is to open Gift. 
__main__


In [5]:

def close():
    """ This is closed """
    print("close gift")
close()


close gift


In [2]:
import functools
def decorater(func):
    @functools.wraps(func)
    def wrapper():
        print("remove wrap")
        func()
        print("There is a toy")
    return wrapper

In [46]:
@decorater
def open():
    """ This is to open Gift. """
    print("open gift")
open()

remove wrap
open gift
There is a toy


In [42]:
print(open.__name__)
print(open.__doc__)
print(open.__module__)

wrapper
None
__main__


In [ ]:
#after wrapping
print(open.__name__)
print(open.__doc__)
print(open.__module__)

open
This is to open Gift. 
__main__


In [9]:
decorater(open)

remove wrap
open gift
There is a toy


In [13]:
gift = decorater(open)

remove wrap
open gift
There is a toy


In [26]:
@decorater
def open():
    print("open gift")
open()


remove wrap
open gift
There is a toy


In [28]:
def logger(func):
    def wrapper(*args, **kwargs):     # ← accept any arguments
        print(f"Calling {func.__name__}")
        return func(*args, **kwargs)  # ← pass them to original
    return wrapper

@logger
def add(a, b):
    return a + b

print(add(3, 5))

Calling add
8


In [29]:
def bold(func):
    def wrapper():
        return "<b>" + func() + "</b>"
    return wrapper

def italic(func):
    def wrapper():
        return "<i>" + func() + "</i>"
    return wrapper

@bold          # ← applied SECOND
@italic        # ← applied FIRST
def say():
    return "Hello"

print(say())    # <b><i>Hello</i></b>

<b><i>Hello</i></b>


In [31]:
def repeat(n):              # ← outer function takes decorator argument
    def decorator(func):    # ← actual decorator
        def wrapper():
            for _ in range(n):
                func()
        return wrapper
    return decorator        # ← returns the decorator

@repeat(3)                  # ← calls repeat(3) which returns decorator
def hello():
    print("Hello!")

hello()

Hello!
Hello!
Hello!


In [48]:
import functools

class MyDecorator:
    def __init__(self, func):         # ① receives the function
        functools.update_wrapper(self, func)
        self.func = func              # stores it as an attribute

    def __call__(self, *args, **kwargs):  # ② called when function runs
        print("Before")
        result = self.func(*args, **kwargs)
        print("After")
        return result


@MyDecorator
def greet():
    print("Hello!")

greet()

Before
Hello!
After


In [50]:
import functools

def my_decorator(func):
    @functools.wraps(func)               # preserves original function identity
    def wrapper(*args, **kwargs):        # works with ANY function signature
        print("Before")
        result = func(*args, **kwargs)   # captures return value
        print("After")
        return result                    # returns it properly
    return wrapper


@my_decorator
def greet():
    print("Hello!")
greet()

Before
Hello!
After


In [52]:
import time
import functools

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        duration = end - start
        print(f"⏱️ '{func.__name__}' took {duration:.4f} seconds")
        return result
    return wrapper


# ① Fast function
@timer
def add(a, b):
    return a + b
add(5,7)

⏱️ 'add' took 0.0000 seconds


12

In [ ]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    database="mydb",
    user="postgres",
    password="password",
    port="5432"
)

print("Connected successfully!")

# always close
conn.close()

In [ ]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    database="mydb",
    user="postgres",
    password="password"
)

cur = conn.cursor()

cur.execute("SELECT id, name FROM users")

rows = cur.fetchall()

for row in rows:
    print(row)

cur.close()
conn.close()

In [53]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    database="mydb",
    user="postgres",
    password="password"
)

cur = conn.cursor()

name = "Khaleel"
age = 25

cur.execute(
    "INSERT INTO users (name, age) VALUES (%s, %s)",
    (name, age)   # ✅ safe
)

conn.commit()   # IMPORTANT

cur.close()
conn.close()

ModuleNotFoundError: No module named 'psycopg2'

In [54]:
import os
from sqlalchemy import create_engine
from sqlalchemy.exc import OperationalError, ArgumentError
from dotenv import load_dotenv

load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")

if not DATABASE_URL:
    raise ValueError("DATABASE_URL not found in .env file")

try:
    engine = create_engine(
        DATABASE_URL,
        echo=False,
        pool_pre_ping=True,
        pool_size=5,
        max_overflow=10,
        pool_timeout=30,
        pool_recycle=1800,
    )
    print(f"✅ Engine created successfully!")
    print(f"   Database: {engine.url.database}")
    print(f"   Host:     {engine.url.host}")
    print(f"   Port:     {engine.url.port}")

except ArgumentError as e:
    print(f"❌ Invalid DATABASE_URL format: {e}")
    raise

ModuleNotFoundError: No module named 'sqlalchemy'

In [55]:
import os
import re
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker, declarative_base
from sqlalchemy.exc import OperationalError, ArgumentError
from dotenv import load_dotenv

load_dotenv()

# ── Helper — mask password for safe printing ──────────────
def mask_url(url: str) -> str:
    return re.sub(r"(:\/\/)([^:]+):([^@]+)(@)", r"\1\2:***\4", url)


# ── Engine ────────────────────────────────────────────────
DATABASE_URL = os.getenv("DATABASE_URL")

if not DATABASE_URL:
    raise ValueError("DATABASE_URL not found in .env file")

engine = create_engine(
    DATABASE_URL,
    echo=False,
    pool_pre_ping=True,
)
print(f"Engine created:    {mask_url(DATABASE_URL)}")


# ── Session Factory ───────────────────────────────────────
SessionLocal = sessionmaker(
    bind=engine,
    autocommit=False,
    autoflush=False,
)
print("Session factory ready.")


# ── Base ──────────────────────────────────────────────────
Base = declarative_base()


# ── Verify Connection ─────────────────────────────────────
def verify_connection():
    try:
        with engine.connect() as conn:

            # ① Basic connectivity check
            result = conn.execute(text("SELECT 1"))
            value  = result.scalar()
            print(f"Connection verified: SELECT 1 returned {value}")

            # ② Check which database we connected to
            db_result = conn.execute(text("SELECT current_database()"))
            db_name   = db_result.scalar()
            print(f"Connected to DB:     {db_name}")

            # ③ Check PostgreSQL version
            ver_result = conn.execute(text("SELECT version()"))
            version    = ver_result.scalar()
            print(f"PostgreSQL version:  {version.split(',')[0]}")

            # ④ Check tables exist
            tables_result = conn.execute(text("""
                SELECT table_name
                FROM information_schema.tables
                WHERE table_schema = 'public'
                ORDER BY table_name
            """))
            tables = [row[0] for row in tables_result]

            if tables:
                print(f"Tables found:        {', '.join(tables)}")
            else:
                print("Tables found:        none yet")

            print("Database connection successful!")

    except OperationalError as e:
        print(f"❌ Connection failed: {e}")
        print("Check: host, port, password, db name in .env")
        raise

    except ArgumentError as e:
        print(f"❌ Invalid DATABASE_URL format: {e}")
        raise


verify_connection()

ModuleNotFoundError: No module named 'sqlalchemy'

In [57]:
var = {"khaleel":"vizag"}
print(var)

{'khaleel': 'vizag'}


In [67]:
class animal:
    def __init__(self,name,type):
        self.name = name
        self.__type = type
    def display(self):
        print(f"Name: {self.name} ,{self.__animal_type}")

In [68]:
dog = animal("john snow","mammal")
dog.display()

AttributeError: 'animal' object has no attribute '_animal__animal_type'

In [71]:
class myclass:
    state = "Karnataka"
    def __init__(self,name,city):
        self.name = name
        self.city = city
    def display(self):
        print(f"Name: {self.name} ,City: {self.city} ,State: {myclass.state}")

In [72]:
f1 = myclass("khaleel","vizag")
f1.display()

Name: khaleel ,City: vizag ,State: Karnataka
